# Exercise 1: Your First ADK Agent

## Learning Objectives

In this exercise, you will:
- Create a conversational agent with ADK
- Understand agent configuration parameters
- Test multi-turn conversations

**Estimated time:** 15 minutes

## Setup Instructions

**⏱️ Estimated time: ~2 minutes**

This exercise assumes you have completed the setup verification notebook and your environment is ready.

If you haven't completed setup yet, please go back to: [00-setup-verification.ipynb](00-setup-verification.ipynb)
**Stuck on an error?** See [TROUBLESHOOTING.md](TROUBLESHOOTING.md) for solutions to common issues.


In [ ]:
# @title Quick Setup (Run this cell first)
# This cell sets up your environment - you can collapse it after running

import os
import google.generativeai as genai
from getpass import getpass

# Step 1: Install Google ADK
!pip install -q google-adk==1.23.0
print("✓ google-adk 1.23.0 installed")

# Step 2: Configure API Key
api_key = getpass('Enter your Google AI API Key: ')
genai.configure(api_key=api_key)
os.environ['GOOGLE_API_KEY'] = api_key
print("✓ API Key configured")

print("\n🚀 Ready to build your first agent!")

---

## What is an AI Agent?

**📖 Estimated time: ~3 minutes**

<!-- INSTRUCTOR: Pause here to explain agent vs chatbot distinction -->

An **AI Agent** is more than just a chatbot. While chatbots simply respond to user messages, agents can:

- **Use tools** to take actions (search databases, call APIs, perform calculations)
- **Access knowledge** from documents and knowledge bases
- **Maintain state** to remember context across conversations
- **Make decisions** about which tools to use and when

In this exercise, we'll start with a **simple conversational agent** - one that can have intelligent conversations but doesn't yet have tools or knowledge. We'll add those capabilities in later exercises.

---

### Agent Components (Conceptual Diagram)

```
┌─────────────────────────────────────┐
│         Your Agent                  │
├─────────────────────────────────────┤
│  Model: gemini-2.5-flash           │  ← Which LLM powers the agent
│  Name: my_first_agent              │  ← Identifier
│  Description: What it does         │  ← Brief summary
│  Instruction: How it behaves       │  ← THE KEY to good agents!
└─────────────────────────────────────┘
```

## The Agent Class

<!-- INSTRUCTOR: Emphasize instruction as the key differentiator -->

In Google ADK, you create agents using the `Agent` class. Every agent needs **4 key parameters**:

### 1. `model` - Which LLM powers your agent
- We'll use `gemini-2.5-flash` - Google's latest fast model
- Other options: `gemini-2.5-pro` (more capable but slower)

### 2. `name` - Identifier for the agent
- Used in logging and deployment
- Should be descriptive and unique

### 3. `description` - Brief summary of agent's purpose
- Helps you remember what this agent does
- Used in documentation and deployment

### 4. `instruction` - System prompt that shapes agent behavior
- **This is where "context engineering" begins!**
- The instruction defines:
  - Agent's personality (formal vs casual, brief vs detailed)
  - Agent's responsibilities (what it should help with)
  - Agent's constraints (what it should NOT do)

> 💡 **Key Insight:** The quality of your agent depends heavily on the quality of your instruction. We'll explore this more as we build.

---

## ✏️ YOUR TURN: Create Your First Agent

**⏱️ Estimated time: ~5 minutes**

Complete the TODO sections below to create your first agent.

**Hint:** Make your agent a helpful travel assistant!

In [ ]:
# Import necessary components
from google.adk.agents import Agent

# Create your agent
agent = Agent(
    model='gemini-2.5-flash',  # The LLM that powers your agent
    name='my_first_agent',      # TODO: Give your agent a name
    description='',             # TODO: Describe what your agent does
    instruction='',             # TODO: Write instructions for your agent
)

print(f"✓ Agent '{agent.name}' created")

In [ ]:
# Import ADK components needed for proper agent testing
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai.types import Content, Part

# Set up session service (manages conversation memory)
session_service = InMemorySessionService()
user_id = "workshop_participant"

print("✓ ADK components imported")
print("✓ Session service created")

# Test your agent using the proper ADK pattern
async def test_agent(agent, query):
    """Run a query against the agent using Runner and Sessions"""
    # Create a new session for this conversation
    session = await session_service.create_session(
        app_name=agent.name,
        user_id=user_id
    )
    
    # Create a runner to execute the agent
    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )
    
    # Run the query and get the response
    final_response = ""
    async for event in runner.run_async(
        user_id=user_id,
        session_id=session.id,
        new_message=Content(parts=[Part(text=query)], role="user")
    ):
        if event.is_final_response():
            final_response = event.content.parts[0].text
            break
    
    return final_response

# Test the agent
query = "Hello! What can you help me with?"
response = await test_agent(agent, query)
print(response)

---

## Understanding the ADK Pattern

<!-- INSTRUCTOR: Walk through Runner + Sessions pattern -->

In ADK, you don't call the agent directly. Instead, you use the **Runner + Sessions** pattern:

### Key Components

1. **Session Service** (`InMemorySessionService`)
   - Manages conversation memory
   - Stores conversation history

2. **Session** (created per conversation)
   - Represents one conversation thread
   - Has a unique `session_id`

3. **Runner** (executes the agent)
   - Takes your agent and runs it
   - Handles the LLM calls
   - Processes tool calls and events

4. **Events** (streaming responses)
   - ADK returns events as the agent thinks
   - You extract the final response with `event.is_final_response()`

### Why This Pattern?

This pattern enables:
- **Multi-turn conversations** with memory
- **Tool calling** with proper event handling
- **Session management** for production deployments
- **Event streaming** to see agent thinking in real-time

### How Instruction Shapes Responses

Notice how your agent's personality and style match your instruction:

- If you wrote "Be brief", responses should be short
- If you wrote "Be enthusiastic", responses should be upbeat
- If you wrote "Focus on travel", responses should relate to travel

> 💡 **Try this:** Change your instruction above and re-run the cells. See how the response changes!

---

## Understanding the Response

<!-- INSTRUCTOR: Walk through a few participant responses -->

The `test_agent()` function returns the agent's response as a string. The response content depends on your agent's instruction.

### How Instruction Shapes Responses

Notice how your agent's personality and style match your instruction:

- If you wrote "Be brief", responses should be short
- If you wrote "Be enthusiastic", responses should be upbeat
- If you wrote "Focus on travel", responses should relate to travel

> 💡 **Try this:** Change your instruction above and re-run the cells. See how the response changes!

## Key Takeaways

Let's review what you learned:

1. ✅ **The Runner + Sessions pattern is fundamental** - ADK uses this pattern (not direct agent calls) for all agent interactions

2. ✅ **The `instruction` parameter is critical** - It defines your agent's personality, knowledge, and behavior

3. ✅ **Sessions enable memory** - By using the SAME session for multiple turns, the agent remembers the conversation context

4. ✅ **Event streaming reveals agent thinking** - You can see events as the agent processes your request

5. ✅ **Preview of what's next:**
   - In Exercise 2, we'll add **tools** so the agent can search for real travel data
   - In Exercise 3, we'll add **knowledge** from a destination guide database  
   - In Exercise 4, we'll add **persistent sessions** so conversations continue even after restarts

In [ ]:
# @title Solution: Exercise 1 - Create Your First Agent (Expand to see)

from google.adk.agents import Agent

# ============================================================
# IMPLEMENTATION
# ============================================================

travel_agent = Agent(
    model='gemini-2.5-flash',
    name='travel_buddy',
    description='A friendly travel planning assistant that helps with trip planning, destination recommendations, and travel tips.',
    instruction='''You are a knowledgeable travel assistant.

Your responsibilities:
- Help users plan trips and answer travel questions
- Provide destination recommendations based on preferences
- Share practical travel tips and cultural insights

Be friendly, concise, and enthusiastic about travel.
If you don't know something, say so rather than guessing.''',
)

print(f"Agent '{travel_agent.name}' created")

# ============================================================
# WHY THIS WORKS
# ============================================================
#
# 1. model='gemini-2.5-flash'
#    - Specifies which Gemini model powers the agent
#    - 'flash' is fast and cost-effective for most use cases
#    - 'pro' is more capable but slower/more expensive
#
# 2. name='travel_buddy'
#    - Identifier used for logging, debugging, and deployment
#    - Should be descriptive and unique within your app
#    - Used in session management (app_name parameter)
#
# 3. description='A friendly travel planning assistant...'
#    - Brief summary of agent's purpose
#    - Helps developers understand the agent's role
#    - Used in documentation and agent registries
#
# 4. instruction='You are a knowledgeable travel assistant...'
#    - THE MOST IMPORTANT PARAMETER - this is context engineering!
#    - Defines personality, responsibilities, and constraints
#    - Structure: Role -> Responsibilities -> Behavior -> Constraints
#
# ============================================================
# KEY INSIGHT
# ============================================================
#
# The 'instruction' parameter is your main control lever for agent
# behavior. A well-structured instruction with clear responsibilities
# and constraints creates predictable, helpful agents. Think of it as
# programming with natural language - you're defining the agent's
# "personality" and decision-making framework.
#
# Pattern: Role + Responsibilities + Behavior Guidelines + Constraints
# ============================================================